# Study 2 check: does the model's yes-score separate inside from outside? (Gemma 3 12B; exploratory)

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
3. **Runtime → Change runtime type → A100 GPU.**

Then **Runtime → Run all**. Expected time on an A100: about 5–10 minutes.

**If Colab disconnects:** reconnect and press **Run all** again. Finished steps are saved in `MyDrive/geomap/results/s2/scores/`
and skipped on the rerun.

In [ ]:
# 1. GPU check
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
assert gb > 35, f"Only {gb:.0f} GB of GPU memory: pick an A100 (40 GB) runtime"

In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
assert os.path.exists("scripts/s2_scores.py"), "Old bundle: upload the new geomap_colab.zip"
print("already saved:", sorted(os.listdir(f"{RES}/s2/scores")) if os.path.exists(f"{RES}/s2/scores") else "nothing yet")

In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' Gemma 3 import)
!pip -q install "transformers==4.57.3" nnsight accelerate
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")
!python -c "from transformers.models.gemma3.modeling_gemma3 import Gemma3ForConditionalGeneration; print('Gemma 3 import OK')"

In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
import sys
sys.modules.pop("geomap", None)  # a stale or wrong "geomap" import must not shadow the code
sys.path.insert(0, "/content/geomap/src")
from geomap.checkpoint import notify
notify("[S2 scores] Colab run started (or resumed)")

In [ ]:
# 5. Study 2 check: re-score the 400 inside questions keeping P(yes) and P(no) (~5-10 min)
!cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/s2_scores.py --model gemma3-12b 2>&1 | grep -v -i warning

In [ ]:
# 6. One zip of the Study 2 score-check results + the results log, on Drive and downloaded here
shutil.make_archive("/content/s2_scores_results", "zip", "/content/geomap", "results/s2/scores")
with zipfile.ZipFile("/content/s2_scores_results.zip", "a") as z:
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
shutil.copy("/content/s2_scores_results.zip", f"{D}/s2_scores_results.zip")
notify("[S2 scores] all done: results in MyDrive/geomap/s2_scores_results.zip")
from google.colab import files
files.download("/content/s2_scores_results.zip")